# Designed double-mutant sampling: does it recover J better than uniform-random sampling?

Follow-up to `MLP_for_anticorrelated_weights.ipynb`'s J-recovery diagnostic: the ProfileMLP
trained on `N=20,000` PURELY UNIFORM RANDOM sequences badly mispredicted the TRUE top-50
viability variants (median 18.3th percentile of its own predicted-log-enrichment distribution), and
`compare_J` showed why -- several position-pairs' pairwise coupling `J` were barely
recovered at all. `MLP_for_profile_only_weights.ipynb` confirmed the mechanism by removing
`J` entirely (perfect recovery once there's no epistasis left to miss).

This notebook tests a DIFFERENT fix: keep the real `F_viab`/`J_viab` landscape (epistasis
still present), but change HOW the training library is sampled. Instead of drawing `N`
sequences fully i.i.d. uniform -- where a given position-pair's specific `(i=a, j=b)`
combination only shows up by incidental co-occurrence, entangled with 5 other random
positions' effects every time it does -- build the library as an explicit, DESIGNED
single-/double-mutant scan around a handful of backgrounds: the same recipe
`extract_effective_F`/`extract_effective_J` already use to PROBE a trained model, used here
to construct the TRAINING data itself. Every `(i, j, a, b)` pairwise cell is then directly,
cleanly represented (background held fixed, only positions `i` and `j` vary), instead of
relying on chance co-occurrence.

Target library size: ~50,000 sequences. `B=6` backgrounds x the FULL single+double-mutant
grid lands at 51,246 -- close enough to 50k, and using the FULL grid (no subsampling) means
no coverage gaps to worry about. Everything downstream (assay simulation, `ProfileMLP`
training, F/J recovery diagnostics) is identical to the sibling notebooks -- only how
`sequences` is built changes.

## 0. Setup

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Must run before the first `import jax` anywhere -- same convention as the sibling notebooks.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from flax import nnx
from typing import Optional
import optax
from tqdm.auto import tqdm

from sequence_classesV1 import *
from analysisV1 import *
from initialize_weights import (
# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
    load_F_viab_aav9_potts, load_J_viab_aav9_potts,
    initialize_anticorrelated_weights,
    NUM_AMINO_ACIDS, NUM_POSITIONS,
)

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

In [ ]:
class ProfileMLP(nnx.Module):
    """
    MLP over the one-hot encoded per-position sequence (L=7 positions x A=20 amino acids
    -> 140 indicator features) -> scalar score. Same building blocks as SimpleMLP
    (Linear + BatchNorm + Dropout + gelu), with input_dim=L*A: one-hot gives the network
    the categorical amino-acid structure for free, so it only has to learn each position's
    per-amino-acid effect -- and any pairwise coupling -- on top of that, instead of also
    having to rediscover the categorical structure from a raw integer index.
    """

    def __init__(self, input_dim: int, hidden_dims: tuple[int, int] = (256, 128),
                 dropout_rate: float = 0.1, *, rngs: nnx.Rngs):
        h1, h2 = hidden_dims
        self.linear1    = nnx.Linear(input_dim, h1, rngs=rngs)
        self.batchnorm1 = nnx.BatchNorm(h1, use_running_average=False, rngs=rngs)
        self.dropout1   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear2    = nnx.Linear(h1, h2, rngs=rngs)
        self.batchnorm2 = nnx.BatchNorm(h2, use_running_average=False, rngs=rngs)
        self.dropout2   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear3    = nnx.Linear(h2, 1, rngs=rngs)

    def __call__(self, x: jax.Array, *, train: bool, rngs: Optional[nnx.Rngs] = None) -> jax.Array:
        x = self.linear1(x)
        x = self.batchnorm1(x, use_running_average=not train)
        x = self.dropout1(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)

        x = self.linear2(x)
        x = self.batchnorm2(x, use_running_average=not train)
        x = self.dropout2(x, deterministic=not train, rngs=rngs)
        x = nnx.gelu(x)

        return self.linear3(x).squeeze(-1)

In [ ]:
@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    """
    x : (batch, L*A) one-hot encoded per-position amino-acid indicators
    y : (batch,) target log enrichment
    """
    def loss_fn(model, rngs):
        y_pred = model(x, train=True, rngs=rngs)
        return jnp.mean((y_pred - y) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    y_pred = model(x, train=False)
    return jnp.mean((y_pred - y) ** 2)


@nnx.scan(in_axes=(nnx.Carry, 0, 0), out_axes=(nnx.Carry, 0))
def train_epoch_scan(carry, xb, yb):
    """Fuses one epoch's mini-batch train_step calls into a single compiled nnx.scan
    instead of a Python for-loop issuing one XLA dispatch per batch."""
    model, optimizer, rngs = carry
    loss = train_step(model, optimizer, xb, yb, rngs)
    return (model, optimizer, rngs), loss

In [ ]:
def train_profile_mlp(X_train, y_train, X_val, y_val, hidden_dims=(128, 64),
                       dropout_rate=0.1, epochs=300, batch_size=256,
                       peak_lr=1e-3, final_lr=1e-5, weight_decay=0,
                       patience=20, seed=0, verbose=True, model=None):
    """
    model : optional existing ProfileMLP to continue training from (warm start) instead of
    initializing a fresh one.
    """
    rngs = nnx.Rngs(seed)
    if model is None:
        model = ProfileMLP(input_dim=X_train.shape[1], hidden_dims=hidden_dims,
                            dropout_rate=dropout_rate, rngs=rngs)

    n_train         = X_train.shape[0]
    steps_per_epoch = max(n_train // batch_size, 1)
    total_steps     = steps_per_epoch * epochs

    lr_schedule_fn = optax.warmup_cosine_decay_schedule(
        init_value=0., peak_value=peak_lr,
        warmup_steps=int(total_steps * 0.1),
        decay_steps=int(total_steps * 0.9),
        end_value=final_lr,
    )
    optimizer = nnx.Optimizer(
        model, optax.adamw(learning_rate=lr_schedule_fn, weight_decay=weight_decay), wrt=nnx.Param
    )

    X_train, y_train = jnp.asarray(X_train), jnp.asarray(y_train)
    X_val,   y_val   = jnp.asarray(X_val),   jnp.asarray(y_val)

    shuffle_key = jax.random.key(seed)
    best_val, best_state, bad_epochs = float("inf"), None, 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in tqdm(range(epochs), desc="Training ProfileMLP", disable=not verbose):
        shuffle_key, perm_key = jax.random.split(shuffle_key)
        perm      = jax.random.permutation(perm_key, n_train)
        batch_idx = perm[: steps_per_epoch * batch_size].reshape(steps_per_epoch, batch_size)

        (model, optimizer, rngs), step_losses = train_epoch_scan(
            (model, optimizer, rngs), X_train[batch_idx], y_train[batch_idx]
        )
        train_loss = float(jnp.mean(step_losses))
        val_loss   = float(eval_step(model, X_val, y_val))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val - 1e-5:
            best_val, bad_epochs = val_loss, 0
            best_state = nnx.state(model)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch} (best val MSE={best_val:.4f})")
                break

    nnx.update(model, best_state)
    return model, history

## 1. The landscape: real `F_viab`/`J_viab` + anticorrelated `F_sel`/`J_sel`

Unchanged from `MLP_for_anticorrelated_weights.ipynb` -- same real AAV9-derived `F_viab`/
`J_viab`, same `F_sel_anti`/`J_sel_anti` at `anticorrelation=0.2` -- so any difference in
recovery quality between that notebook and this one is attributable ONLY to the sampling
strategy, not to a different landscape.

In [ ]:
# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
F_viab = load_F_viab_aav9_potts()
J_viab = load_J_viab_aav9_potts()

anticorrelation = 0.2
F_sel_anti, J_sel_anti = initialize_anticorrelated_weights(jax.random.key(10), F_viab, J_viab, anticorrelation)

print(f"F_viab shape: {F_viab.shape}  J_viab shape: {J_viab.shape}")

<div style="background:#eaf2fb; border-left:5px solid #2f6fed; border-radius:4px; padding:10px 16px; margin:10px 0; font-size:0.95em; color:#0a2f5c;">
<b>Which "top 500" / top-K is this?</b><br>
This section builds a <b>DESIGNED combinatorial library</b> (category e, not a ranked top-K): <code>B=50</code> random backgrounds x every single mutant x the FULL double-mutant grid (21 position-pairs x 20x20 amino-acid pairs) = 427,050 sequences (<code>sequences</code>/<code>dataset_designed</code>, cached <code>diversity427050_..._designed50k.csv</code>). It is not scored/ranked — it exhaustively covers every pairwise (i,j,a,b) cell around random backgrounds so training data directly represents them, replacing the uniform-random 20k library the sibling notebooks use. (Same construction as the "With a good dataset" section of <code>MLP_bilinear_head_anticorrelated.ipynb</code> — but verified here in section 4 to be genuinely wired into training, unlike there.)
</div>

## 2. Build the designed ~50k-sequence library

`B=6` random backgrounds, then for each: the background itself, every single mutant
(`L*A = 140`), and the FULL double-mutant grid (`n_pairs*A*A = 21*400 = 8,400`, no
subsampling -- every pairwise cell gets direct coverage from every background). Total =
`B * (1 + 140 + 8,400)`.

In [ ]:
def build_designed_library(backgrounds, L=NUM_POSITIONS, A=NUM_AMINO_ACIDS):
    """
    Same single-/double-mutant grid as extract_effective_F/extract_effective_J, but used
    here to build the TRAINING sequence pool itself instead of only probing an
    already-trained model. For each background: the background itself, every single mutant,
    and every double mutant (FULL grid) -- so every (i, j, a, b) pairwise cell is directly,
    cleanly represented in the training data (background held fixed, only i and j vary),
    instead of relying on incidental co-occurrence in a random pool. Duplicate sequence
    values are fine -- ProtocolV3 treats each row as an independent pool slot regardless
    (self.d0 = sequences.shape[0]), same as everywhere else in this codebase.
    """
    backgrounds = np.asarray(backgrounds, dtype=np.int64)
    B       = backgrounds.shape[0]
    pairs   = [(i, j) for i in range(L) for j in range(i + 1, L)]
    n_pairs = len(pairs)

    # Singles.
    n_single    = B * L * A
    singles     = np.repeat(backgrounds, L * A, axis=0)
    pos_pattern = np.tile(np.repeat(np.arange(L), A), B)
    aa_pattern  = np.tile(np.tile(np.arange(A), L), B)
    singles[np.arange(n_single), pos_pattern] = aa_pattern

    # Doubles (full grid).
    pairs_arr = np.array(pairs)
    grid_pair = np.repeat(np.arange(n_pairs), A * A)
    grid_a    = np.tile(np.repeat(np.arange(A), A), n_pairs)
    grid_b    = np.tile(np.tile(np.arange(A), A), n_pairs)
    grid_i    = pairs_arr[grid_pair, 0]
    grid_j    = pairs_arr[grid_pair, 1]

    n_double  = B * n_pairs * A * A
    doubles   = np.repeat(backgrounds, n_pairs * A * A, axis=0)
    pos_i_all = np.tile(grid_i, B)
    pos_j_all = np.tile(grid_j, B)
    a_all     = np.tile(grid_a, B)
    b_all     = np.tile(grid_b, B)
    row_idx   = np.arange(n_double)
    doubles[row_idx, pos_i_all] = a_all
    doubles[row_idx, pos_j_all] = b_all

    return np.concatenate([backgrounds, singles, doubles], axis=0)


key = jax.random.key(0)
key, k_bg = jax.random.split(key)

B = 50  # backgrounds -- B * (1 + L*A + n_pairs*A*A) lands right at the ~50k target
backgrounds_train = jax.random.randint(k_bg, shape=(B, NUM_POSITIONS), minval=0, maxval=NUM_AMINO_ACIDS)

sequences  = build_designed_library(backgrounds_train)
n_pairs    = NUM_POSITIONS * (NUM_POSITIONS - 1) // 2

print(f"backgrounds: {B}")
print(f"singles    : {B * NUM_POSITIONS * NUM_AMINO_ACIDS:,}")
print(f"doubles    : {B * n_pairs * NUM_AMINO_ACIDS * NUM_AMINO_ACIDS:,}  "
      f"(FULL grid, {n_pairs} pairs x {NUM_AMINO_ACIDS}x{NUM_AMINO_ACIDS})")
print(f"TOTAL designed library size: {sequences.shape[0]:,}")

<div style="background:#eaf2fb; border-left:5px solid #2f6fed; border-radius:4px; padding:10px 16px; margin:10px 0; font-size:0.95em; color:#0a2f5c;">
<b>Which "top 500" / top-K is this?</b><br>
Same DESIGNED library from section 2 (<code>sequences</code>, 427,050 rows) — this cell only runs the <code>ProtocolV3</code> assay simulation on it to attach real log-enrichment targets (<code>dataset_designed</code>). No new population.
</div>

## 3. Simulate the assay on the designed library

Same `ProtocolV3` construction and parameters as the sibling notebooks (`T_viab=T_sel=1`,
`noise_viab=noise_sel=0.5`, `dilution_factor=10`, `N0=1e9`, `N1=5e8`) -- only `sequences`
differs.

In [ ]:
protocol_designed = ProtocolV3(multinomialNGS=True, N0=1_000_000_000, N1=500_000_000,
        dilution_factor=10, sequences=sequences, D=1e9,
        F_viab=F_viab, J_viab=J_viab, F_sel=F_sel_anti, J_sel=J_sel_anti,
        noise_viab=0.5, noise_sel=0.5, T_sel=1, T_viab=1,
        )

rounds = protocol_designed.N_loop_DE(1)
bio_row, ngs_row = rounds[0]
lambda0p, lambda2p, lambda3p = (np.asarray(a) for a in ngs_row)

eps = 1.0  # pseudocount, same convention as the sibling notebooks

log_enr_viab = np.log((lambda2p + eps) / (lambda0p + eps))  # target1 -- viability
log_enr_sel  = np.log((lambda3p + eps) / (lambda2p + eps))  # target2 -- selectivity

print(f"log_enr_viab: min={log_enr_viab.min():.2f}  max={log_enr_viab.max():.2f}  mean={log_enr_viab.mean():.2f}")
print(f"log_enr_sel : min={log_enr_sel.min():.2f}  max={log_enr_sel.max():.2f}  mean={log_enr_sel.mean():.2f}")

In [ ]:
def dataset_filename(protocol, label):
    def fmt(v):
        return f"{float(v):g}".replace(".", "")

    if protocol.noise_viab == protocol.noise_sel:
        noise_part = f"noise{fmt(protocol.noise_viab)}"
    else:
        noise_part = f"noiseviab{fmt(protocol.noise_viab)}_noisesel{fmt(protocol.noise_sel)}"

    ngs_part = "multinomial" if protocol.multinomialNGS else "nbinom"

    return (f"diversity{protocol.d0}_Tsel{fmt(protocol._T_sel)}"
            f"_Tviab{fmt(protocol._T_viab)}_{noise_part}_{ngs_part}_{label}.csv")


def build_or_load_dataset(protocol, sequences, log_enr_viab, log_enr_sel, label):
    """Same recipe as the sibling notebooks -- `label` keeps this notebook's CSVs from
    colliding with the correlated/anticorrelated/independent/profile_only ones, which share
    the same T_sel/T_viab/noise parameters but different diversity and/or sampling."""
    path = dataset_filename(protocol, label)
    if os.path.exists(path):
        print(f"{path} already exists -- loading from disk")
        return pd.read_csv(path)

    seq_strings = ["".join(AA_LABELS[a] for a in row) for row in np.asarray(sequences)]
    df = pd.DataFrame({"sequence": seq_strings, "target1": log_enr_viab, "target2": log_enr_sel})
    df.to_csv(path, index=False)
    print(f"saved {path} ({len(df)} rows)")
    return df


dataset_designed = build_or_load_dataset(protocol_designed, sequences, log_enr_viab, log_enr_sel, label="designed50k")
dataset_designed.head()

## 4. Train ProfileMLP on the designed library

In [ ]:
def split_train_val(X, y, val_frac=0.15, seed=0):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    n_val = int(len(X) * val_frac)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


@nnx.jit
def predict_log_enrichment(model, x):
    return model(x, train=False)


### ------------ Sequence string -> raw indices -> one-hot ------------ ###
lut = np.zeros(256, dtype=np.int64)
for i, aa in enumerate(AA_LABELS):
    lut[ord(aa)] = i

seq_bytes  = np.frombuffer("".join(dataset_designed["sequence"]).encode("ascii"), dtype=np.uint8)
seq_matrix = lut[seq_bytes].reshape(len(dataset_designed), NUM_POSITIONS)
seq_oh     = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[seq_matrix].reshape(len(dataset_designed), -1)

y_viab_all = dataset_designed["target1"].to_numpy()
y_sel_all  = dataset_designed["target2"].to_numpy()

idx_train, idx_test = train_test_split(np.arange(len(dataset_designed)), test_size=0.5, random_state=0)
X_train_full, X_test = seq_oh[idx_train], seq_oh[idx_test]

print(f"X_train_full shape: {X_train_full.shape}  X_test shape: {X_test.shape}")

In [ ]:
y_viab_train, y_viab_test = y_viab_all[idx_train], y_viab_all[idx_test]
Xtr_viab, ytr_viab, Xva_viab, yva_viab = split_train_val(X_train_full, y_viab_train, val_frac=0.15, seed=0)
print(f"ProfileMLP (viab) -- train: {len(Xtr_viab):,}  val: {len(Xva_viab):,}  test: {len(X_test):,}")

model_viab, hist_viab = train_profile_mlp(Xtr_viab, ytr_viab, Xva_viab, yva_viab, seed=0)

In [ ]:
y_sel_train, y_sel_test = y_sel_all[idx_train], y_sel_all[idx_test]
Xtr_sel, ytr_sel, Xva_sel, yva_sel = split_train_val(X_train_full, y_sel_train, val_frac=0.15, seed=1)
print(f"ProfileMLP (sel) -- train: {len(Xtr_sel):,}  val: {len(Xva_sel):,}  test: {len(X_test):,}")

model_sel, hist_sel = train_profile_mlp(Xtr_sel, ytr_sel, Xva_sel, yva_sel, seed=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, hist, name in [(axes[0], hist_viab, "viab (target1)"), (axes[1], hist_sel, "sel (target2)")]:
    ax.plot(hist["train_loss"], label="train MSE")
    ax.plot(hist["val_loss"],   label="val MSE")
    ax.set_xlabel("epoch"); ax.set_ylabel("MSE")
    ax.set_title(f"ProfileMLP ({name}) training curve")
    ax.legend(); ax.grid(True, linestyle="--", alpha=0.4)
fig.tight_layout()
plt.show()

pred_viab_test = np.asarray(predict_log_enrichment(model_viab, jnp.asarray(X_test)))
pred_sel_test  = np.asarray(predict_log_enrichment(model_sel,  jnp.asarray(X_test)))

print(f"target1 (viab) -- Pearson r (test): {pearson(y_viab_test, pred_viab_test):.4f}")
print(f"target2 (sel)  -- Pearson r (test): {pearson(y_sel_test,  pred_sel_test):.4f}")

## 5. F recovery

Uses FRESH independent random backgrounds for the recovery probe -- NOT the 6 backgrounds
used to build the training library, and NOT rows sampled from `dataset_designed` itself
(those are all clustered within 2 mutations of just 6 points, which would make recovery
look artificially good by only testing right where the model was trained hardest). 256
fresh uniform-random backgrounds, same count as the sibling notebooks, for a fair,
apples-to-apples comparison.

In [ ]:
def compare_F(F_viab, F_sel, title, label_a="F_viab", label_b="F_sel"):
    """Same 3-panel recipe as the sibling notebooks' F comparisons: two heatmaps
    (shared RdBu_r scale) + a raw-entry scatter with Pearson r."""
    F_viab, F_sel = np.asarray(F_viab), np.asarray(F_sel)
    vmax = max(np.abs(F_viab).max(), np.abs(F_sel).max())

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, F, panel_title in [(axes[0], F_viab, label_a), (axes[1], F_sel, label_b)]:
        im = ax.imshow(F, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.set_title(panel_title)
        ax.set_xlabel("Position"); ax.set_ylabel("Amino acid")
        ax.set_xticks(range(NUM_POSITIONS)); ax.set_xticklabels(range(1, NUM_POSITIONS + 1))
        ax.set_yticks(range(NUM_AMINO_ACIDS)); ax.set_yticklabels(AA_LABELS)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    r = pearson(F_viab.ravel(), F_sel.ravel())
    axes[2].scatter(F_viab.ravel(), F_sel.ravel(), s=14, alpha=0.6)
    lims = [min(F_viab.min(), F_sel.min()), max(F_viab.max(), F_sel.max())]
    axes[2].plot(lims, lims, "k--", alpha=0.5)
    axes[2].set_xlabel(label_a); axes[2].set_ylabel(label_b)
    axes[2].set_title(f"Pearson r = {r:.3f}")

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def extract_effective_F(model, backgrounds, L=NUM_POSITIONS, A=NUM_AMINO_ACIDS, batch_size=8192):
    """Single-mutant scan against background sequences -- same recipe as the sibling
    notebooks' extract_effective_F."""
    backgrounds = np.asarray(backgrounds, dtype=np.int64)
    M = backgrounds.shape[0]

    n_single       = M * L * A
    single_mutants = np.repeat(backgrounds, L * A, axis=0)
    pos_pattern    = np.tile(np.repeat(np.arange(L), A), M)
    aa_pattern     = np.tile(np.tile(np.arange(A), L), M)
    single_mutants[np.arange(n_single), pos_pattern] = aa_pattern

    x_all    = np.concatenate([backgrounds, single_mutants], axis=0)
    x_all_oh = np.eye(A, dtype=np.float32)[x_all].reshape(x_all.shape[0], -1)

    scores_chunks = []
    for start in range(0, x_all_oh.shape[0], batch_size):
        chunk = jnp.asarray(x_all_oh[start : start + batch_size])
        scores_chunks.append(np.asarray(predict_log_enrichment(model, chunk)))
    scores = np.concatenate(scores_chunks)

    mean_score    = scores.mean()
    single_scores = scores[M:].reshape(M, L, A)

    F_flat = single_scores.mean(axis=0) - mean_score
    return F_flat.T.astype(np.float32)


key_probe = jax.random.key(999)  # independent of the training-library backgrounds
probe_backgrounds = np.asarray(
    jax.random.randint(key_probe, shape=(256, NUM_POSITIONS), minval=0, maxval=NUM_AMINO_ACIDS)
)

F_viab_hat = extract_effective_F(model_viab, probe_backgrounds)
F_sel_hat  = extract_effective_F(model_sel,  probe_backgrounds)

compare_F(F_viab, F_viab_hat, "F_viab (ground truth) vs ProfileMLP-recovered F_viab_hat [designed 50k library]",
          label_a="F_viab", label_b="F_viab_hat")
compare_F(F_sel_anti, F_sel_hat, "F_sel (ground truth) vs ProfileMLP-recovered F_sel_hat [designed 50k library]",
          label_a="F_sel", label_b="F_sel_hat")

## 6. J recovery -- the actual test

Same `extract_effective_J`/`compare_J` diagnostic as `MLP_for_anticorrelated_weights.ipynb`,
same `probe_backgrounds`. This is the number that matters: does the per-pair recovery
Pearson r (the bar chart, sorted worst-to-best) improve over the uniform-random 20k run?

In [ ]:
def extract_effective_J(model, backgrounds, L=NUM_POSITIONS, A=NUM_AMINO_ACIDS, batch_size=8192):
    """
    Double-mutant scan: for each background and each position-pair (i, j), scores every
    (a, b) joint mutation, then isolates j_{ij}(a, b) via the double difference
    M(s^{i->a,j->b}) - M(s^{i->a}) - M(s^{j->b}) + M(s). Same recipe as the sibling
    notebook's extract_effective_J.
    """
    backgrounds = np.asarray(backgrounds, dtype=np.int64)
    M       = backgrounds.shape[0]
    pairs   = [(i, j) for i in range(L) for j in range(i + 1, L)]
    n_pairs = len(pairs)

    def score_all(seqs):
        seqs_oh = np.eye(A, dtype=np.float32)[seqs].reshape(seqs.shape[0], -1)
        chunks = []
        for start in range(0, seqs_oh.shape[0], batch_size):
            chunk = jnp.asarray(seqs_oh[start : start + batch_size])
            chunks.append(np.asarray(predict_log_enrichment(model, chunk)))
        return np.concatenate(chunks)

    n_single       = M * L * A
    single_mutants = np.repeat(backgrounds, L * A, axis=0)
    pos_pattern    = np.tile(np.repeat(np.arange(L), A), M)
    aa_pattern     = np.tile(np.tile(np.arange(A), L), M)
    single_mutants[np.arange(n_single), pos_pattern] = aa_pattern

    base_scores   = score_all(backgrounds)
    single_scores = score_all(single_mutants).reshape(M, L, A)

    pairs_arr = np.array(pairs)
    grid_pair = np.repeat(np.arange(n_pairs), A * A)
    grid_a    = np.tile(np.repeat(np.arange(A), A), n_pairs)
    grid_b    = np.tile(np.tile(np.arange(A), A), n_pairs)
    grid_i    = pairs_arr[grid_pair, 0]
    grid_j    = pairs_arr[grid_pair, 1]

    n_double       = M * n_pairs * A * A
    double_mutants = np.repeat(backgrounds, n_pairs * A * A, axis=0)
    pos_i_all      = np.tile(grid_i, M)
    pos_j_all      = np.tile(grid_j, M)
    a_all          = np.tile(grid_a, M)
    b_all          = np.tile(grid_b, M)
    row_idx        = np.arange(n_double)
    double_mutants[row_idx, pos_i_all] = a_all
    double_mutants[row_idx, pos_j_all] = b_all

    double_scores = score_all(double_mutants).reshape(M, n_pairs, A, A)

    J_hat = np.zeros((L, L, A, A), dtype=np.float32)
    for k, (i, j) in enumerate(pairs):
        single_i = single_scores[:, i, :][:, :, None]
        single_j = single_scores[:, j, :][:, None, :]
        contrast = double_scores[:, k] - single_i - single_j + base_scores[:, None, None]
        J_hat[i, j] = contrast.mean(axis=0)
        J_hat[j, i] = J_hat[i, j].T

    return J_hat


def compare_J(J_true, J_hat, title, label_a="J", label_b="J_hat", L=NUM_POSITIONS):
    """Same J-recovery diagnostic as the sibling notebook: global scatter (r + std ratio),
    per-pair Pearson r bar chart, and best-/worst-recovered pair heatmaps."""
    J_true, J_hat = np.asarray(J_true), np.asarray(J_hat)
    pairs = [(i, j) for i in range(L) for j in range(i + 1, L)]

    true_flat, hat_flat, pair_r = [], [], []
    for (i, j) in pairs:
        t, h = J_true[i, j].ravel(), J_hat[i, j].ravel()
        true_flat.append(t); hat_flat.append(h)
        pair_r.append(pearson(t, h))
    true_flat   = np.concatenate(true_flat)
    hat_flat    = np.concatenate(hat_flat)
    pair_r      = np.array(pair_r)
    pair_labels = [f"{i}-{j}" for i, j in pairs]
    order       = np.argsort(pair_r)
    worst_k, best_k = order[0], order[-1]

    fig = plt.figure(figsize=(16, 8))
    gs  = fig.add_gridspec(2, 4, height_ratios=[1.1, 1])

    ax_sc = fig.add_subplot(gs[0, :2])
    ax_sc.scatter(true_flat, hat_flat, s=5, alpha=0.12)
    lims = [min(true_flat.min(), hat_flat.min()), max(true_flat.max(), hat_flat.max())]
    ax_sc.plot(lims, lims, "k--", alpha=0.5, label="y = x")
    r_all = pearson(true_flat, hat_flat)
    ax_sc.set_xlabel(label_a); ax_sc.set_ylabel(label_b)
    ax_sc.set_title(f"All {len(true_flat):,} off-diagonal entries -- r = {r_all:.3f}  "
                     f"(std ratio {label_b}/{label_a} = {hat_flat.std() / true_flat.std():.2f})")
    ax_sc.legend(fontsize=8)

    ax_bar = fig.add_subplot(gs[0, 2:])
    colors = ["crimson" if k == worst_k else ("seagreen" if k == best_k else "steelblue") for k in order]
    ax_bar.bar(range(len(pairs)), pair_r[order], color=colors)
    ax_bar.set_xticks(range(len(pairs)))
    ax_bar.set_xticklabels([pair_labels[k] for k in order], rotation=90, fontsize=7)
    ax_bar.axhline(0, color="black", lw=0.8)
    ax_bar.set_ylabel("Pearson r"); ax_bar.set_xlabel("position pair, sorted by recovery quality")
    ax_bar.set_title(f"{label_b} recovery quality per position-pair")

    for col_offset, k, tag in [(0, worst_k, "WORST"), (2, best_k, "BEST")]:
        i, j = pairs[k]
        vmax = max(np.abs(J_true[i, j]).max(), np.abs(J_hat[i, j]).max())
        for sub, (arr, name) in enumerate([(J_true[i, j], label_a), (J_hat[i, j], label_b)]):
            ax = fig.add_subplot(gs[1, col_offset + sub])
            im = ax.imshow(arr, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
            ax.set_title(f"{tag} pair {i}-{j}\n{name} (r={pair_r[k]:.2f})", fontsize=9)
            ax.set_xlabel(f"aa @ pos {j}"); ax.set_ylabel(f"aa @ pos {i}")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(title)
    fig.tight_layout()
    return dict(zip(pair_labels, pair_r))


J_viab_hat = extract_effective_J(model_viab, probe_backgrounds)
J_sel_hat  = extract_effective_J(model_sel,  probe_backgrounds)

pair_r_viab = compare_J(J_viab, J_viab_hat, "J_viab (ground truth) vs ProfileMLP-recovered J_viab_hat [designed 50k library]",
                         label_a="J_viab", label_b="J_viab_hat")
plt.show()
pair_r_sel = compare_J(J_sel_anti, J_sel_hat, "J_sel (ground truth) vs ProfileMLP-recovered J_sel_hat [designed 50k library]",
                        label_a="J_sel", label_b="J_sel_hat")
plt.show()

print("Per-pair recovery (Pearson r), viability:  ", {k: round(v, 2) for k, v in pair_r_viab.items()})
print("Per-pair recovery (Pearson r), selectivity:", {k: round(v, 2) for k, v in pair_r_sel.items()})

<div style="background:#eaf2fb; border-left:5px solid #2f6fed; border-radius:4px; padding:10px 16px; margin:10px 0; font-size:0.95em; color:#0a2f5c;">
<b>Which "top 500" / top-K is this?</b><br>
This is the <b>GLOBAL/THEORETICAL top-50</b>: an exhaustive brute-force scan (<code>brute_force_top_k</code>, <code>K_BEST=50</code>) over all <code>20**7</code> ≈ 1.28 billion possible sequences, scored against the ground-truth F/J (<code>best_seqs_viab</code>/<code>best_seqs_sel</code>) — identical recipe and K to the sibling notebooks. Distinct from the DESIGNED 427,050-sequence library (section 2) used to train the models tested against it here; the very next cell (<code>16545798</code>) directly checks overlap between the two and finds zero.
</div>

## 7. Do the TRUE global-optimum variants get recognized by the MLP now?

Full circle back to the original question. Same exhaustive brute-force scan
(`compute_score_array`/`_score_chunk`/`brute_force_top_k`) over the full `20**7` sequence
space as the sibling notebooks, against the SAME real `F_viab`/`J_viab` and
`F_sel_anti`/`J_sel_anti` used everywhere in this notebook -- so `best_scores_viab`/
`best_scores_sel` below should match the sibling notebook's numbers (35.48 / 19.39) up to
RNG.

In [ ]:
def compute_score_array(seq, F, J, L=NUM_POSITIONS):
    """Same formula as Protocol.compute_score, decoupled from a Protocol instance so it can
    be applied to an arbitrary (n, L) array of raw-index sequences."""
    scores = jnp.sum(F[seq, jnp.arange(L)], axis=1)
    for i in range(L):
        for j in range(i + 1, L):
            scores = scores + J[i, j, seq[:, i], seq[:, j]]
    return scores


@jax.jit
def _score_chunk(idx, F, J, L=NUM_POSITIONS, A=NUM_AMINO_ACIDS):
    tmp = idx
    digits = []
    for _ in range(L):
        digits.append(tmp % A)
        tmp = tmp // A
    seq = jnp.stack(digits[::-1], axis=1).astype(jnp.int32)
    return seq, compute_score_array(seq, F, J, L)


def brute_force_top_k(F, J, k=50, chunk_size=20_000_000, L=NUM_POSITIONS, A=NUM_AMINO_ACIDS):
    """Exhaustively scores EVERY possible sequence (A**L of them) against the ground-truth F/J
    and keeps a running top-k, in chunks (so the full A**L x L array of sequences is never
    materialized at once)."""
    total = A ** L
    best_scores = jnp.full((k,), -jnp.inf)
    best_seqs   = jnp.zeros((k, L), dtype=jnp.int32)

    for start in tqdm(range(0, total, chunk_size), desc="brute-force scan"):
        end = min(start + chunk_size, total)
        idx = jnp.arange(start, end)
        seq_chunk, scores_chunk = _score_chunk(idx, F, J)

        kk = min(k, scores_chunk.shape[0])
        local_vals, local_idx = jax.lax.top_k(scores_chunk, kk)
        local_seqs = seq_chunk[local_idx]

        cand_scores = jnp.concatenate([best_scores, local_vals])
        cand_seqs   = jnp.concatenate([best_seqs, local_seqs], axis=0)
        best_scores, merge_idx = jax.lax.top_k(cand_scores, k)
        best_seqs = cand_seqs[merge_idx]

    return np.asarray(best_seqs), np.asarray(best_scores)


K_BEST = 50
best_seqs_viab, best_scores_viab = brute_force_top_k(F_viab, J_viab, k=K_BEST)
best_seqs_sel,  best_scores_sel  = brute_force_top_k(F_sel_anti, J_sel_anti, k=K_BEST)

print(f"TRUE global-optimum viability score: {best_scores_viab[0]:.2f}")
print(f"TRUE global-optimum selectivity score: {best_scores_sel[0]:.2f}")

In [ ]:
def seqs_to_strings(seq_array):
    return ["".join(AA_LABELS[a] for a in row) for row in np.asarray(seq_array)]

best_viab_strings = seqs_to_strings(best_seqs_viab)
best_sel_strings  = seqs_to_strings(best_seqs_sel)

sampled_set  = set(dataset_designed["sequence"])
n_viab_found = sum(s in sampled_set for s in best_viab_strings)
n_sel_found  = sum(s in sampled_set for s in best_sel_strings)

print(f"Of the top-{K_BEST} TRUE-best viability variants, {n_viab_found} appear in the "
      f"{len(dataset_designed):,}-sequence designed pool")
print(f"Of the top-{K_BEST} TRUE-best selectivity variants, {n_sel_found} appear in the "
      f"{len(dataset_designed):,}-sequence designed pool")

<div style="background:#eaf2fb; border-left:5px solid #2f6fed; border-radius:4px; padding:10px 16px; margin:10px 0; font-size:0.95em; color:#0a2f5c;">
<b>Which "top 500" / top-K is this?</b><br>
Two populations here: (1) the GLOBAL/THEORETICAL top-50 reused from section 7 (<code>best_seqs_viab</code>/<code>best_seqs_sel</code>); and (2) a fresh <b>RANDOM SYNTHETIC BASELINE</b> — 200,000 new random sequences (<code>sequences_bg</code>/<code>dataset_designed_bg</code>/<code>X_bg</code>, cached <code>diversity200000_..._designed50k_bg.csv</code>), never used in training, built purely to serve as the background reference distribution for the percentile calculation (analogous to the 2M-sample <code>X_big</code> background in the bilinear-head notebooks, just a different N here).
</div>

## 8. MLP-predicted percentile of the TRUE top-50

Same percentile analysis as the sibling notebooks, against a background pool of 200,000
FRESH random sequences (never used in training) run through the same `F`/`J` landscape --
this is the headline number to compare against the uniform-random notebook's 18.3th
(viability) / 100.0th (selectivity) median percentiles.

In [ ]:
key_bg = jax.random.key(42)
key_bg, k_seq_bg = jax.random.split(key_bg)

N_BG = 200_000
sequences_bg = jax.random.randint(k_seq_bg, shape=(N_BG, NUM_POSITIONS), minval=0, maxval=NUM_AMINO_ACIDS)

protocol_designed_bg = ProtocolV3(multinomialNGS=True, N0=1_000_000_000, N1=500_000_000,
        dilution_factor=10, sequences=sequences_bg, D=1e9,
        F_viab=F_viab, J_viab=J_viab, F_sel=F_sel_anti, J_sel=J_sel_anti,
        noise_viab=0.5, noise_sel=0.5, T_sel=1, T_viab=1,
        )

rounds_bg = protocol_designed_bg.N_loop_DE(1)
_, ngs_row_bg = rounds_bg[0]
lambda0p_bg, lambda2p_bg, lambda3p_bg = (np.asarray(a) for a in ngs_row_bg)

log_enr_viab_bg = np.log((lambda2p_bg + eps) / (lambda0p_bg + eps))
log_enr_sel_bg  = np.log((lambda3p_bg + eps) / (lambda2p_bg + eps))

dataset_designed_bg = build_or_load_dataset(
    protocol_designed_bg, sequences_bg, log_enr_viab_bg, log_enr_sel_bg, label="designed50k_bg"
)

seq_bytes_bg  = np.frombuffer("".join(dataset_designed_bg["sequence"]).encode("ascii"), dtype=np.uint8)
seq_matrix_bg = lut[seq_bytes_bg].reshape(len(dataset_designed_bg), NUM_POSITIONS)
X_bg          = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[seq_matrix_bg].reshape(len(dataset_designed_bg), -1)

pred_viab_bg = np.asarray(predict_log_enrichment(model_viab, jnp.asarray(X_bg)))
pred_sel_bg  = np.asarray(predict_log_enrichment(model_sel,  jnp.asarray(X_bg)))

viab_scores_bg = np.asarray(compute_score_array(jnp.asarray(seq_matrix_bg), F_viab, J_viab))
sel_scores_bg  = np.asarray(compute_score_array(jnp.asarray(seq_matrix_bg), F_sel_anti, J_sel_anti))

print(f"TRUE global optimum viability: {best_scores_viab[0]:.2f}  (best of {N_BG:,} random: {viab_scores_bg.max():.2f})")
print(f"TRUE global optimum selectivity: {best_scores_sel[0]:.2f}  (best of {N_BG:,} random: {sel_scores_bg.max():.2f})")

In [ ]:
X_best_viab = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[best_seqs_viab].reshape(K_BEST, -1)
X_best_sel  = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[best_seqs_sel].reshape(K_BEST, -1)

pred_best_viab = np.asarray(predict_log_enrichment(model_viab, jnp.asarray(X_best_viab)))
pred_best_sel  = np.asarray(predict_log_enrichment(model_sel,  jnp.asarray(X_best_sel)))

pct_viab = [100 * (pred_viab_bg < v).mean() for v in pred_best_viab]
pct_sel  = [100 * (pred_sel_bg  < v).mean() for v in pred_best_sel]

print(f"MLP-predicted log enrichment for the TRUE #1 viability variant: {pred_best_viab[0]:.2f}  "
      f"-> {pct_viab[0]:.1f}th percentile of the {N_BG:,}-sample predicted-log-enrichment distribution")
print(f"MLP-predicted log enrichment for the TRUE #1 selectivity variant: {pred_best_sel[0]:.2f}  "
      f"-> {pct_sel[0]:.1f}th percentile of the {N_BG:,}-sample predicted-log-enrichment distribution")
print(f"Median percentile across all top-{K_BEST} TRUE-best viability variants: {np.median(pct_viab):.1f}")
print(f"Median percentile across all top-{K_BEST} TRUE-best selectivity variants: {np.median(pct_sel):.1f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred_pop, pred_best, name in [(axes[0], pred_viab_bg, pred_best_viab, "viability"),
                                       (axes[1], pred_sel_bg,  pred_best_sel,  "selectivity")]:
    ax.hist(pred_pop, bins=80, color="lightgray",
            label=f"predicted log enrichment, {len(pred_pop):,}-variant random sample")
    for v in pred_best:
        ax.axvline(v, color="crimson", alpha=0.4, lw=1)
    ax.axvline(pred_best[0], color="crimson", lw=2, label=f"TRUE top-{K_BEST} global-optimum variants")
    ax.set_xlabel("MLP-predicted log enrichment")
    ax.set_ylabel("count")
    ax.set_title(f"[designed 50k] Where do the TRUE-best {name} variants land in the MLP's predicted distribution?")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, true_pop, true_best, name in [(axes[0], viab_scores_bg, best_scores_viab, "viability"),
                                       (axes[1], sel_scores_bg,  best_scores_sel,  "selectivity")]:
    ax.hist(true_pop, bins=80, color="lightgray",
            label=f"TRUE score, {len(true_pop):,}-variant random sample")
    for v in true_best:
        ax.axvline(v, color="darkorange", alpha=0.4, lw=1)
    ax.axvline(true_best[0], color="darkorange", lw=2, label=f"TRUE top-{K_BEST} global-optimum variants")
    ax.set_xlabel("TRUE ground-truth score")
    ax.set_ylabel("count")
    ax.set_title(f"[designed 50k] Where do the TRUE top-{K_BEST} {name} variants land in the true score distribution?")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Conclusion

**Fill in after running.** Compare against `MLP_for_anticorrelated_weights.ipynb` (uniform
random, `N=20,000`):

| | uniform-random 20k | designed 50k (this notebook) |
|---|---|---|
| median percentile, viability top-50 | 18.3th | ? |
| median percentile, selectivity top-50 | 100.0th | ? |
| worst-recovered `J_viab` pair (r) | ? | ? |
| best-recovered `J_viab` pair (r) | ? | ? |

- If the designed library's median viability percentile is meaningfully higher and the
  per-pair recovery bars in §6 are more uniformly high (less spread between worst and best
  pair), that confirms explicit double-mutant coverage is the right lever, and it's worth
  pushing further (more backgrounds, or iterating -- new backgrounds centered on THIS run's
  own top predictions, closing the directed-evolution loop for real).
- If it's barely different, the bottleneck isn't combinatorial coverage of pairs but
  something else (e.g. the network's inductive bias still doesn't isolate pairwise terms
  well even when they're cleanly represented in the data) -- worth trying the explicit
  bilinear/factorized architecture next instead of pushing sampling further.